In [ ]:
# ============================================================
# SVM-BASED PRIORITIZATION OF TOP 50 METABOLIC REACTIONS
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.gridspec import GridSpec

# ============================================================
# 2. INPUT FILE
# ============================================================

input_file = "/Users/apple/Downloads/Script_CBM_and_Analysis/4.Analysis/KS_Test/Consensus_Reactions_Pathways.csv"

output_top50 = "Top50_Reactions.csv"

# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(input_file)

print("Total reactions :", len(df))

# ============================================================
# 4. KEEP ONLY SIGNIFICANT REACTIONS
# ============================================================

df = df[df["Reject"] == True].copy()

print("Significant reactions :", len(df))

# ============================================================
# 5. DATA CLEANING
# ============================================================

df["Pvalue"] = df["Pvalue"].replace(0,1e-300)
df["Padj"]   = df["Padj"].replace(0,1e-300)

df = df.replace([np.inf,-np.inf],np.nan)
df = df.dropna()

# ============================================================
# 6. CREATE FEATURES
# ============================================================

df["logP"] = -np.log10(df["Pvalue"])
df["logPadj"] = -np.log10(df["Padj"])

X = df[["FC","logP","logPadj"]]

# Dummy labels
# Replace with actual labels if available

y = (df["FC"] > 0).astype(int)

# ============================================================
# 7. STANDARDIZE FEATURES
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ============================================================
# 8. TRAIN LINEAR SVM
# ============================================================

svm = SVC(
    kernel="linear",
    probability=True,
    random_state=42
)

svm.fit(X_scaled,y)

print("SVM Training Completed")

# ============================================================
# 9. FEATURE IMPORTANCE
# ============================================================

coef = np.abs(svm.coef_[0])

feature_importance = pd.DataFrame({

    "Feature":X.columns,
    "Weight":coef

}).sort_values(
    by="Weight",
    ascending=False
)

print("\nFeature Importance")
print(feature_importance)

# ============================================================
# 10. REACTION IMPORTANCE
# ============================================================

reaction_score = svm.predict_proba(X_scaled)[:,1]

df["SVMImportance"] = reaction_score

# ============================================================
# 11. NORMALIZATION
# ============================================================

def normalize(x):

    return (x-x.min())/(x.max()-x.min()+1e-9)

df["norm_SVM"] = normalize(df["SVMImportance"])

df["norm_logPadj"] = normalize(df["logPadj"])

df["norm_FC"] = normalize(np.abs(df["FC"]))

# ============================================================
# 12. FINAL COMPOSITE SCORE
# ============================================================

w1 = 0.4
w2 = 0.3
w3 = 0.3

df["FinalScore"] = (

      w1*df["norm_SVM"]
    + w2*df["norm_logPadj"]
    + w3*df["norm_FC"]

)

# ============================================================
# 13. TOP 50 REACTIONS
# ============================================================

top50 = (

    df.sort_values(
        by="FinalScore",
        ascending=False
    )
    .head(50)
    .copy()

)

top50.to_csv(
    output_top50,
    index=False
)

print("\nTop 50 reactions saved.")

# ============================================================
# 14. FEATURES FOR CLUSTERING
# ============================================================

cluster_data = top50[
    [
        "FC",
        "logPadj",
        "FinalScore"
    ]
]

cluster_data = (

    cluster_data-cluster_data.mean()

)/(

    cluster_data.std()+1e-9

)

# ============================================================
# 15. HIERARCHICAL CLUSTERING
# ============================================================

Z = linkage(
    cluster_data,
    method="ward"
)

# ============================================================
# 16. FIGURE
# ============================================================

fig = plt.figure(figsize=(10,14))

gs = GridSpec(
    1,
    2,
    width_ratios=[1.5,1],
    wspace=0.05
)

# ------------------------------------------------------------
# DENDROGRAM
# ------------------------------------------------------------

ax1 = fig.add_subplot(gs[0])

d = dendrogram(

    Z,

    labels=top50["Reaction"].values,

    orientation="left",

    leaf_font_size=8,

    ax=ax1

)

ax1.set_xlabel("Distance")

ax1.set_title("Reaction Clustering")

# ------------------------------------------------------------
# ORDER
# ------------------------------------------------------------

ordered = top50.iloc[d["leaves"]]

# ------------------------------------------------------------
# BUBBLE PLOT
# ------------------------------------------------------------

ax2 = fig.add_subplot(gs[1])

bubble = normalize(
    ordered["FinalScore"]
)

ypos = np.arange(len(ordered))

sc = ax2.scatter(

    ordered["FinalScore"],

    ypos,

    s=bubble*700+50,

    c=ordered["FinalScore"],

    cmap="Reds",

    alpha=0.9

)

ax2.set_yticks(ypos)

ax2.set_yticklabels([])

ax2.set_xlabel("Composite Importance Score")

ax2.set_title("Reaction Importance")

cbar = plt.colorbar(
    sc,
    ax=ax2
)

cbar.set_label("Importance")

plt.tight_layout()

plt.savefig(
    "Figure3A_Top50_Reactions.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 17. SUMMARY
# ============================================================

print("\n=======================================")
print("TOP 50 REACTION ANALYSIS COMPLETED")
print("=======================================")

print("Output Files")
print("------------")
print("1. Top50_Reactions.csv")
print("2. Figure3A_Top50_Reactions.png")
print("=======================================")